> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# Final test: the paper's rule and the best trials, on data never used

**The owner asked for this run on Nov 17, 2026.** Nothing was changed after the development work: the six rows are the ones of `winfut_best_vs_paper.ipynb` (the paper's rule, two variants of it, and the best trial of the 50,000-trial search by WIN, by WDO and by the joint score), with the same costs (B3 fee plus one tick a side), the same sizing and the same code. The best trials are re-selected here from the saved development trials, so no final-period number can influence the choice.

**How the final period is scored.** The strategy runs over the development bars **and** the final bars as one history (the bands and the volatility need the days before 2025-01-02, and the equity path continues), and only the days from **2025-01-02** are scored. The sizing is relative to the running equity, so the returns are comparable with the development ones.

**Rule written before the run** (brief section 10, a proposal the owner has not yet approved, so it is reported, not applied as a verdict): net of costs, the return beats the CDI over the same window and the daily t-statistic exceeds 1.65. Sharpe, drawdown and the year-by-year table are reported.

**Benchmarks end on 2026-10-16** (the project's CDI and IBOV files); the strategy's data runs to 2026-10-30. The strategy statistics use all final days; the `vs CDI` and `vs IBOV` columns use the strategy's CAGR over the days up to 2026-10-16, so the windows match.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from b3core.benchmark import compare_with_cdi, compare_with_index
from b3core.intraday_momentum import extended_performance, performance
from b3core.intraday_search import INSTRUMENTS, TrialSet, decode, run_trial, scores
from b3core.winfut import FINAL_START, load_bars

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 230
pd.options.display.max_columns = 40
pd.options.display.max_colwidth = 100

N_TRIALS, SEED, SLIPPAGE_TICKS, MIN_TRADES_FULL = 50_000, 20261116, 1.0, 300
saved = np.load(Path("../samples/optimization") / f"trials_{N_TRIALS}_seed{SEED}.npz")
dev_days = pd.DatetimeIndex(saved["days"])
dev = TrialSet(indices=saved["indices"], days=dev_days,
               ret={n: saved[f"ret_{n}"] for n in ("WIN", "WDO")},
               sides={n: saved[f"sides_{n}"] for n in ("WIN", "WDO")})

FINAL_DAY0 = pd.Timestamp(FINAL_START.date())
bars = {}
for name, inst in INSTRUMENTS.items():
    development = load_bars("../samples", symbol=inst.symbol)
    final = load_bars("../samples", symbol=inst.symbol, final=True)
    assert development.index.max() < FINAL_DAY0 <= final.index.min()
    bars[name] = pd.concat([development, final])
all_days = pd.DatetimeIndex(sorted(set.intersection(*[set(f.index.normalize().unique()) for f in bars.values()])))
final_mask = np.asarray(all_days >= FINAL_DAY0)
final_days = all_days[final_mask]
print(f"{len(all_days)} days in all; final period {final_days[0].date()} to {final_days[-1].date()}, {len(final_days)} days")
print({n: len(f) for n, f in bars.items()}, "bars (development + final)")

1228 days in all; final period 2025-01-02 to 2026-10-30, 458 days
{'WIN': 681919, 'WDO': 688511} bars (development + final)


In [2]:
def clock(minutes: int) -> str:
    return "session end" if minutes >= 1440 else f"{minutes // 60:02d}:{minutes % 60:02d}"


def describe(p: dict) -> str:
    return (f"{p['lookback']}d x{p['band_mult']:g} every {p['trade_freq']}m from {clock(p['first_decision'])} "
            f"exit {clock(p['exit_minute'])} {p['stop']}/{p['stop_check']} {p['sides']}")


PAPER = {"lookback": 14, "band_mult": 1.0, "trade_freq": 30, "first_decision": 540, "exit_minute": 1440,
         "stop": "vwap_band", "stop_check": "grid", "sides": "both"}
EVERYTHING = np.ones(len(dev_days), dtype=bool)
chosen = {"paper's rule": PAPER, "paper, exit at 17:30": PAPER | {"exit_minute": 1050},
          "paper, every-bar stop": PAPER | {"stop_check": "every_bar"}}
for mode in ("WIN", "WDO", "joint"):
    s = scores(dev, EVERYTHING, mode, MIN_TRADES_FULL)
    chosen[f"best by {mode}"] = decode(int(dev.indices[int(np.nanargmax(s))]))
display(pd.Series({k: describe(v) for k, v in chosen.items()}, name="combination").to_frame())

,combination
paper's rule,14d x1 every 30m from 09:00 exit session end vwap_band/grid both
"paper, exit at 17:30",14d x1 every 30m from 09:00 exit 17:30 vwap_band/grid both
"paper, every-bar stop",14d x1 every 30m from 09:00 exit session end vwap_band/every_bar both
best by WIN,20d x1.5 every 60m from 12:00 exit 18:20 current_band/every_bar both
best by WDO,14d x1 every 15m from 09:30 exit 18:20 vwap_only/grid long
best by joint,20d x1.5 every 15m from 10:30 exit 18:20 vwap_only/grid both


## 1. The run

In [3]:
daily, trades = {}, {}
for label, params in chosen.items():
    returns, sides = run_trial(bars, params, all_days, slippage_ticks=SLIPPAGE_TICKS)
    flat = {n: pd.Series(np.nan_to_num(r, nan=0.0), index=all_days) for n, r in returns.items()}
    daily[label] = flat | {"50/50": (flat["WIN"] + flat["WDO"]) / 2.0}
    trades[label] = {n: pd.Series(s, index=all_days) for n, s in sides.items()}
# the development part of the run must reproduce the development search (same code, longer history)
row = int(np.where(dev.indices == int(dev.indices[int(np.nanargmax(scores(dev, EVERYTHING, "joint", MIN_TRADES_FULL)))]))[0][0])
check = daily["best by joint"]["WIN"][~final_mask].to_numpy()
print("development part against the saved search: max abs difference", float(np.abs(check - np.nan_to_num(dev.ret["WIN"][row], nan=0.0)).max()))

development part against the saved search: max abs difference 0.0


## 2. Benchmarks

In [4]:
ibov = pd.read_csv("../samples/ifr2_universe/ibov.csv", index_col=0, parse_dates=True)["Close"].dropna()
cdi_raw = pd.read_csv("../samples/ifr2_universe/cdi.csv")
cdi = pd.Series(cdi_raw["rate"].to_numpy(), index=pd.to_datetime(cdi_raw["date"], format="%d/%m/%Y"))
LAST_BENCH = min(ibov.index.max(), cdi.index.max())
bench_days = final_days[final_days <= LAST_BENCH]
years = (bench_days[-1] - bench_days[0]).days / 365.25
cdi_cmp, ibov_cmp = compare_with_cdi(0.0, years, cdi, bench_days), compare_with_index(0.0, years, ibov, bench_days)
print(f"benchmark window {bench_days[0].date()} to {bench_days[-1].date()} ({years:.2f} years): CDI {cdi_cmp.cdi_cagr:.2%} a year, "
      f"buying the IBOV {ibov_cmp.cagr:.2%} a year")

benchmark window 2025-01-02 to 2026-10-16 (1.79 years): CDI 14.32% a year, buying the IBOV 35.07% a year


## 3. The final test

One table per instrument and one for the 50/50 average (each day's return is the mean of the WIN account's and the WDO account's return: half the capital each, rebalanced daily, each account sized on its own equity).

In [5]:
def metrics_row(returns: pd.Series) -> dict:
    base, extra = performance(returns), extended_performance(returns)
    covered = extended_performance(returns[returns.index <= LAST_BENCH])
    cdi_vs = compare_with_cdi(covered["cagr"], covered["years"], cdi, bench_days)
    ibov_vs = compare_with_index(covered["cagr"], covered["years"], ibov, bench_days)
    return {
        "total return %": base["total_return"] * 100, "CAGR %": extra["cagr"] * 100, "Sharpe": base["sharpe"],
        "Sortino": extra["sortino"], "Calmar": extra["calmar"], "profit factor": extra["profit_factor"],
        "Max DD %": extra["max_drawdown"] * 100, "hit ratio %": base["hit_ratio"] * 100, "t-stat": base["t_stat"],
        "vs CDI, pts": cdi_vs.excess * 100, "beats CDI": "yes" if cdi_vs.beats_cdi else "no",
        "vs IBOV, pts": ibov_vs.excess * 100, "beats IBOV": "yes" if ibov_vs.beats else "no",
    }


final_tables = {}
for series in ("WIN", "WDO", "50/50"):
    final_tables[series] = pd.DataFrame({k: metrics_row(daily[k][series][final_mask]) for k in chosen}).T
    print(f"=== {series}, final test ===")
    display(final_tables[series])

=== WIN, final test ===


,total return %,CAGR %,Sharpe,Sortino,Calmar,profit factor,Max DD %,hit ratio %,t-stat,"vs CDI, pts",beats CDI,"vs IBOV, pts",beats IBOV
paper's rule,45.48,22.82,1.18,2.33,1.28,1.30,17.80,41.03,1.59,9.37,yes,-11.37,no
"paper, exit at 17:30",45.96,23.04,1.19,2.39,1.32,1.31,17.49,42.07,1.61,10.29,yes,-10.45,no
"paper, every-bar stop",59.02,28.97,1.56,3.65,2.76,1.45,10.50,36.01,2.10,15.27,yes,-5.47,no
best by WIN,42.61,21.49,2.10,5.55,3.82,2.15,5.62,43.38,2.83,7.81,yes,-12.93,no
best by WDO,20.48,10.76,0.73,1.44,0.64,1.22,16.93,40.84,0.98,-3.52,no,-24.26,no
best by joint,34.84,17.82,1.06,1.83,1.39,1.34,12.80,44.85,1.43,5.48,yes,-15.26,no


=== WDO, final test ===


,total return %,CAGR %,Sharpe,Sortino,Calmar,profit factor,Max DD %,hit ratio %,t-stat,"vs CDI, pts",beats CDI,"vs IBOV, pts",beats IBOV
paper's rule,-16.97,-9.69,-0.48,-0.87,-0.49,0.90,19.97,33.05,-0.65,-22.78,no,-43.52,no
"paper, exit at 17:30",-20.79,-12.00,-0.67,-1.19,-0.57,0.86,21.01,32.28,-0.90,-25.42,no,-46.16,no
"paper, every-bar stop",-30.99,-18.40,-1.12,-2.03,-0.59,0.78,31.16,26.50,-1.51,-32.35,no,-53.09,no
best by WIN,6.60,3.57,0.38,0.71,0.31,1.15,11.54,35.71,0.52,-10.39,no,-31.14,no
best by WDO,-28.82,-17.01,-1.09,-1.70,-0.59,0.72,28.88,34.43,-1.47,-28.44,no,-49.19,no
best by joint,-5.02,-2.79,-0.10,-0.16,-0.13,0.97,22.18,39.41,-0.13,-13.71,no,-34.45,no


=== 50/50, final test ===


,total return %,CAGR %,Sharpe,Sortino,Calmar,profit factor,Max DD %,hit ratio %,t-stat,"vs CDI, pts",beats CDI,"vs IBOV, pts",beats IBOV
paper's rule,11.10,5.94,0.46,0.91,0.40,1.10,14.87,36.83,0.62,-7.27,no,-28.02,no
"paper, exit at 17:30",8.54,4.60,0.38,0.75,0.33,1.08,13.93,36.76,0.51,-8.51,no,-29.26,no
"paper, every-bar stop",5.80,3.14,0.30,0.62,0.28,1.06,11.31,31.22,0.40,-10.68,no,-31.43,no
best by WIN,23.68,12.36,1.48,3.17,1.61,1.58,7.66,39.17,1.99,-1.46,no,-22.21,no
best by WDO,-6.30,-3.51,-0.27,-0.45,-0.30,0.94,11.85,39.33,-0.36,-16.16,no,-36.90,no
best by joint,14.07,7.49,0.60,1.05,0.73,1.16,10.33,38.64,0.81,-4.05,no,-24.80,no


## 4. The rule, and development against final

`passes the rule`: the return beats the CDI **and** the daily t-statistic is above 1.65. Then, for each row, the development Sharpe next to the final Sharpe: the gap is the optimism of the selection.

In [6]:
rule = pd.DataFrame({s: [(t["beats CDI"] == "yes") & (t["t-stat"] > 1.65) for _, t in [(0, final_tables[s])]][0] for s in final_tables})
rule = rule.map(lambda v: "yes" if v else "no")
print("Passes the proposed rule (beats the CDI and t > 1.65):")
display(rule)

dev_sharpe = pd.DataFrame({s: {k: performance(daily[k][s][~final_mask])["sharpe"] for k in chosen} for s in ("WIN", "WDO", "50/50")})
final_sharpe = pd.DataFrame({s: final_tables[s]["Sharpe"] for s in final_tables})
gap = pd.concat({"development Sharpe": dev_sharpe, "final Sharpe": final_sharpe, "final minus development": final_sharpe - dev_sharpe}, axis=1)
display(gap)

Passes the proposed rule (beats the CDI and t > 1.65):


,WIN,WDO,50/50
paper's rule,no,no,no
"paper, exit at 17:30",no,no,no
"paper, every-bar stop",yes,no,no
best by WIN,yes,no,no
best by WDO,no,no,no
best by joint,no,no,no


development Sharpe             final Sharpe             final minus development            
                                     WIN   WDO 50/50          WIN   WDO 50/50                     WIN   WDO 50/50
paper's rule                        0.46  0.44  0.58         1.18 -0.48  0.46                    0.72 -0.93 -0.12
paper, exit at 17:30               -0.29  0.13 -0.10         1.19 -0.67  0.38                    1.48 -0.79  0.47
paper, every-bar stop               0.52 -0.59 -0.06         1.56 -1.12  0.30                    1.04 -0.53  0.35
best by WIN                         2.05 -0.79  0.71         2.10  0.38  1.48                    0.05  1.17  0.77
best by WDO                        -0.11  1.57  1.06         0.73 -1.09 -0.27                    0.84 -2.66 -1.33
best by joint                       1.18  1.12  1.48         1.06 -0.10  0.60                   -0.12 -1.22 -0.88

## 5. Year by year, and the trades

In [7]:
years_rows = {}
for k in chosen:
    for s in ("WIN", "WDO", "50/50"):
        r = daily[k][s][final_mask]
        for y, part in r.groupby(r.index.year):
            years_rows[(k, s, y)] = {"return %": ((1 + part).prod() - 1) * 100, "Sharpe": performance(part)["sharpe"], "days": len(part)}
display(pd.DataFrame(years_rows).T.unstack(level=2).round(2))

trade_counts = pd.DataFrame({k: {n: trades[k][n][final_mask].sum() / 2 for n in ("WIN", "WDO")} for k in chosen}).T
trade_counts.columns = [f"{c} round trips" for c in trade_counts.columns]
display(trade_counts)

return %        Sharpe         days       
                                2025   2026   2025  2026   2025   2026
best by WDO           50/50     0.03  -6.34   0.06 -0.71 250.00 208.00
                      WDO     -11.66 -19.43  -0.63 -1.80 250.00 208.00
                      WIN      11.68   7.88   0.78  0.66 250.00 208.00
best by WIN           50/50    18.02   4.80   2.04  0.76 250.00 208.00
                      WDO      10.18  -3.25   1.04 -0.29 250.00 208.00
                      WIN      26.08  13.11   2.40  1.71 250.00 208.00
best by joint         50/50    20.05  -4.99   1.33 -0.46 250.00 208.00
                      WDO      16.81 -18.69   1.02 -1.59 250.00 208.00
                      WIN      22.33  10.23   1.21  0.85 250.00 208.00
paper's rule          50/50    16.53  -4.66   1.06 -0.35 250.00 208.00
                      WDO      -2.37 -14.95  -0.04 -1.06 250.00 208.00
                      WIN      37.62   5.71   1.78  0.45 250.00 208.00
paper, every-bar stop 50/50    14.62  -7.69   1.01 -0.75 250.00 208.00
                      WDO      -4.11 -28.03  -0.15 -2.46 250.00 208.00
                      WIN      35.73  17.16   1.84  1.20 250.00 208.00
paper, exit at 17:30  50/50    11.77  -2.89   0.79 -0.19 250.00 208.00
                      WDO      -9.49 -12.48  -0.47 -0.95 250.00 208.00
                      WIN      36.67   6.80   1.73  0.52 250.00 208.00

,WIN round trips,WDO round trips
paper's rule,487.00,575.00
"paper, exit at 17:30",471.00,553.00
"paper, every-bar stop",741.00,865.00
best by WIN,204.00,232.00
best by WDO,267.00,254.00
best by joint,222.00,259.00


## 6. Equity of the 50/50 average

Compounded daily returns from 1 at the start of the final period, against the same money in the CDI and the IBOV.

In [8]:
import plotly.graph_objects as go

fig = go.Figure()
for k in chosen:
    r = daily[k]["50/50"][final_mask]
    fig.add_trace(go.Scatter(x=r.index, y=(1 + r).cumprod(), name=k))
cdi_path = (1 + cdi.reindex(final_days).fillna(0.0) / 100.0).cumprod()
fig.add_trace(go.Scatter(x=cdi_path.index, y=cdi_path / cdi_path.iloc[0], name="CDI", line=dict(color="black", dash="dash")))
bench_ibov = ibov.reindex(final_days).ffill()
fig.add_trace(go.Scatter(x=final_days, y=bench_ibov / bench_ibov.iloc[0], name="IBOV", line=dict(color="grey", dash="dot")))
fig.update_layout(title="Final test, 50/50 average of WIN and WDO (the CDI and the IBOV stop at the last day of their files)", height=420, margin=dict(t=60, b=20))
fig.show()